# プロンプト設計と回答の信頼性

Zero-shot、Few-shot、CoT の使い方を学び、質問の文脈や評価方法が回答にどう影響するかを確認します。

# Zero-shot / Few-shot 



## References

 [Brown2020gpt3] Brown, T. B. et al., Language Models are Few-Shot Learners, NeurIPS 2020. DOI: 10.48550/arXiv.2005.14165

[Liu2021goodexamples] Liu, J. et al., What Makes Good In-Context Examples for GPT-3?, arXiv, 2021. DOI: 10.48550/arXiv.2101.06804.

[Ma2023fairnessfewshot] Ma, H. et al., Fairness-guided Few-shot Prompting for Large Language Models, arXiv, 2023. DOI: 10.48550/arXiv.2303.13217.

### 1. 定義

#### ■ Zero-shot


* 例：

  * 「この文章を要約せよ」
  * 「反応物と生成物を抽出せよ」

👉 モデルの事前学習知識だけで解かせる



#### ■ Few-shot

少数の**入出力例（デモ）を与えてから実行させる方法**

* 例：

```text
入力：A + B → C
出力：reactants=[A,B], products=[C]

入力：D + E → F
出力：reactants=[D,E], products=[F]

入力：X + Y → Z
出力：
```

👉 パターンを見せて「やり方」を学習させる



### 2. 違い（本質）

| 観点  | Zero-shot | Few-shot  |
|---|---|---|
| 指示  | 自然言語のみ    | 例＋指示      |
| 安定性 | 低め        | 高い        |
| 柔軟性 | 高い        | やや制約あり    |
| コスト | 低い        | 高い（トークン増） |



### 3. 使い分け

#### ■ Zero-shotが適する

* 単純なタスク
* モデルがよく知っている領域
* 試作・探索段階

例：

* 要約
* 翻訳
* 一般的なQA



#### ■ Few-shotが適する

* 出力形式が厳密
* タスクが曖昧
* 独自ルールがある

例：

* 論文からの構造化抽出（どういう「構造」かを指定）
* コード生成時の制約（どう書くか、変数名の規則）
* JSONフォーマット固定（キーの固定）



### 4. Few-shotの効果

Few-shotを入れると：

* 出力フォーマットが安定する
* 解釈のブレが減る
* 暗黙ルールを伝えられる

👉 特に「LLMが誤解しやすいタスク」で強力



### 5. 注意点

#### ■ Zero-shotの問題

* 曖昧な指示に弱い
* 出力形式が崩れやすい



#### ■ Few-shotの問題

* トークンコスト増
* 例に引っ張られる（バイアス）
* 不適切な例＝性能悪化



### 6. 現代的なベストプラクティス

単純に二択ではなく：

👉 **Zero-shot + 制約 + 出力仕様**
または
👉 **Few-shot（最小限） + 明確なフォーマット**

が最適






## まとめ（本質）

* Zero-shot：指示だけで解かせる（軽いが不安定）
* Few-shot：例でやり方を教える（重いが安定）

👉 実務では：

> **「曖昧ならFew-shot、明確ならZero-shot」**



---



# Chain-of-Thought（CoT）誘導




## Chain-of-Thoughtとは

まず、
**Chain-of-Thought（CoT）**とは、LLMに問題を解かせるときに、答えだけでなく、中間の考え方・推論手順を文章として出させる方法です。

日本語ではよく、
```
思考の連鎖
段階的推論
途中式つきの reasoning
```
のように説明されます。


### Chain-of-Thought型とは？

まず、Chain-of-Thought型とは？

通常の質問
```
太郎はリンゴを3個持っています。
花子から2個もらいました。
さらに1個食べました。
残りはいくつですか？
```

に対して
```
順を追って考えてください。
```
を追加する手法。


出力：

- 最初に3個あります。
- 2個もらうので、3 + 2 = 5個です。
- 1個食べるので、5 - 1 = 4個です。
- 答えは4個です。

このように、途中の推論を明示するのがChain-of-Thoughtです。

古いモデルでは、次のような指示が有効でした。
```
step by stepで考えてください。
```
あるいは、
```
まず条件を整理し、次に候補を比較し、最後に答えを出してください。
```
こうすると、モデルが問題を分解しやすくなり、正答率が上がることがありました。


### 1. 定義

**Chain-of-Thought（CoT）誘導**とは、
モデルに対して「段階的に推論させる」ことを促すプロンプト技法。

* 例：

  * 「ステップごとに考えよ」
  * 「理由を説明しながら答えよ」
  * 「まず考察し、次に結論を述べよ」

👉 モデル内部の推論を**明示的に言語化させる**ことが本質



### 2. 目的

主な目的は以下：

* 複雑な問題の分解
* 推論の正確性向上
* 計算・論理ミスの低減

特に有効な領域：

* 数学問題
* 論理推論
* 多段階意思決定



### 3. 種類

#### (1) Zero-shot CoT

例：

* 「step-by-stepで考えよ」

👉 最も簡単だが効果は限定的



#### (2) Few-shot CoT

* 推論付きの例を提示する

例：

```
Q: 2+3は？
A: 2と3を足すと5。よって5。
```

👉 精度は大きく向上




### 4. メリット

* 推論精度の向上
* 解の根拠が可視化される
* 複雑問題に強い



### 5. デメリット（重要）

* 出力が冗長になる
* 誤った推論をもっともらしく説明する（ハルシネーション強化）
* フォーマット崩壊（JSONなどと相性が悪い）
* コスト増（トークン増加）



### 6. 現代モデルでの位置づけ（重要）

最新のLLMでは：

👉 **内部的にCoT的推論はすでに行われている**

そのため：

* 明示的に「考えよ」と書かなくてもよい場合が多い
* むしろ書かない方が安定するケースがある




### 7. 代替アプローチ（重要）

CoTの代わりに使うべきもの：

* 出力形式の厳密指定
* 制約の明示

  * 「推測禁止」
  * 「不明はnull」
* 入力範囲の限定

👉 **「思考を指示」ではなく「出力を制御」する**





## まとめ（本質）

* CoTは「推論を言語化させる技法」
* 以前は重要だったが、現在は必須ではない
* 構造化タスクでは逆効果になることが多い

👉 現代の最適戦略：

> **推論はモデルに任せ、出力仕様と制約を厳密に設計する**




---



# ■ ChatGPT 5.5プロンプトの書き方（基本構造）

以下はchatGPT 5.3に尋ねた chatGPT 5.5プロンプトの書き方を示します。



## 1. 役割（自分が何者かを書く）

最初に「誰として振る舞うか」を明示する。

例：

* あなたは材料インフォマティクスの研究者です。
* あなたは化学学科の大学教授です。学生に優しく・丁寧に教えることができます。

👉 これにより、**出力の専門性・視点・粒度が安定**する




## 2. タスク（何をさせたいか）

やってほしいことを具体的に書く。

例：

* 以下の論文の要旨を2文でまとめよ
* 以下の論文から、反応物と生成物のペアを抽出せよ

👉 抽象的に書くとブレるので、「出力形式」まで指定するのが重要





## 3. 条件・制約

出力のルールを固定する。

例：

* 日本語で書け
* 箇条書きで出力せよ
* 初心者向けに説明せよ
* DOIを必ず含めよ

👉 ここが**再現性の鍵**




# ■ Zero-shot / Few-shot の使い分け



## ● Zero-shot Prompt（例なし）

指示だけで解かせる方法

例：

```
あなたは材料科学の研究者です。
以下の文章を2文で要約せよ。
```

👉 特徴

* シンプル・高速
* ただし曖昧になりやすい




## ● Few-shot Prompt（例あり）

「お手本」を見せる方法

例：

```
あなたは材料科学の研究者です。

以下の形式で要約せよ。

例：
入力：XXX
出力：YYY

入力：以下の文章
〜本文〜　（XXX, YYYを具体的に含む）
```

👉 特徴

* 出力フォーマットが安定
* スタイル・粒度を制御できる




# ■ 実用テンプレ（推奨）

```
あなたは〇〇の専門家です。

以下のタスクを実行せよ：
- 〇〇する

制約：
- 日本語
- 箇条書き
- 〇〇を含める

（必要ならFew-shot例）
```




# ■ 重要なポイント（本質）

* 「役割」＝思考の枠を決める
* 「タスク」＝やることを固定する
* 「制約」＝出力を安定させる
* 「例」＝曖昧さを消す


**GPT-5.5向けのプロンプトは「細かく書くほど良い」から「シンプルに意図を渡す」へ変わっている**。
従来のような“手順を縛るプロンプト”はむしろ性能を下げることがある。



---



## 文脈に基づく次トークン予測

LLM は、それまでのトークンを条件に次のトークンの確率分布を計算します。質問が答えの存在を前提にしていると、その前提に沿った続きを生成することがあります。根拠が不足している場合や、そもそも問題が解けない場合でも、もっともらしい答えが出ることがあります。仕組みは [Transformer と文章生成](8300.0140.transformer_and_generation.ipynb) を参照してください。

### 解があると仮定してしまう例

人間が：

```text id="jlwmcz"
物質Aの物性値を正確に教えてしてください
```

と言う。

しかし本来：

```text id="jlwmd0"
学習内容の中に解は存在しない
```

可能性が高い。

それでも LLM は：

* 予測
* 推測
* plausible な説明

を生成してしまう。



回答の事実が根拠と食い違う場合は**ハルシネーション**、ユーザーの見解や前提に合わせて判断を曲げる場合は**迎合（sycophancy）**として考えられます。たとえば誤った前提に同意したうえで根拠のない説明を作る回答には、両方が現れます。

# ハルシネーションの一つの原因

ハルシネーションは単にモデルが悪いからではなく、現在の評価・ベンチマークの採点設計が、分からないと答えるよりも推測して答えることを報酬化している」というものです。つまり、score設計／評価設計が“自信ありげな当てずっぽう”を誘導している、という主張です。

OpenAI公式記事
https://openai.com/index/why-language-models-hallucinate

元論文PDF
https://cdn.openai.com/pdf/d04913be-3f6f-4d2b-b283-ff432ef4aaa5/why-language-models-hallucinate.pdf




## 1. 何が「score設計のせい」なのか

OpenAIの主張は、単純化するとこうです。

> 多くのベンチマークでは、
> **正解なら1点、不正解なら0点、分からないと言っても0点**
> という採点になっている。

この場合、モデルにとっては、分からないときに

```text
分かりません
```

と答えるより、

```text
たぶん○○です
```

と推測した方が得になります。

なぜなら、推測すれば偶然当たって点が入る可能性がある一方で、「分かりません」は最初から0点になるからです。

OpenAIの元論文でも、現在の多くの評価では「分からない」と言うことが罰せられ、推測することが得になるため、モデルが常に試験中のような状態になる、と説明されています。([OpenAI][1])



## 2. 人間の試験で考えると分かりやすい

たとえば、四択問題で分からない問題があるとします。

採点がこうなら：

```text
正解：1点
不正解：0点
無回答：0点
```

合理的な受験者は、分からなくても答えます。

なぜなら、四択なら25%の確率で当たるからです。

つまり、この採点方式では、

```text
正直に「分からない」と言う人
```

よりも、

```text
当てずっぽうでも答える人
```

の方が高得点になりやすい。

OpenAIが言っているのは、LLM評価でも同じことが起きている、という話です。



## 3. LLMではこれがハルシネーションになる

LLMの場合、「分からないが、もっともらしく答える」は次のような形になります。

```text
存在しない論文名を出す
架空のURLを出す
実在しない著者名を出す
日付をそれらしく作る
根拠が弱いのに断定する
```

これがハルシネーションです。

重要なのは、モデルが「嘘をつきたい」わけではないことです。

むしろ、評価設計が

```text
分からないと答える → 点にならない
推測して答える → 当たれば点になる
```

という構造になっているため、**不確実性を表明するよりも、答えを出し切る行動が強化される**ということです。



## 4. OpenAIの主張の核心

OpenAIの記事では、よくある誤解の

```text
モデルの精度が上がれば、ハルシネーションは自然になくなる
```

という考えを否定しています。

理由は、現実には次のような質問があるからです。

```text
そもそも答えが存在しない質問
情報が不足している質問
学習データからは確定できない低頻度の事実
最新情報が必要な質問
曖昧で条件が足りない質問
```

このような問いでは、正しい振る舞いは

```text
分かりません
情報が足りません
確認が必要です
この条件では断定できません
```

です。

ところが、通常の正解率ベースの評価では、こうした「控えめな回答」が十分に報酬されない。OpenAI記事は、ハルシネーション評価を少し追加するだけでは不十分で、主要な評価指標自体を「不確実性の表明を報酬する」形に作り直す必要がある、と述べています。([OpenAI][2])



## 5. 「score設計」が悪いとはどういう意味か

悪いscore設計は、たとえばこうです。

```text
正解：1点
不正解：0点
分からない：0点
```

この設計では、モデルは分からなくても答える方が有利です。

より望ましい設計は、たとえば次のようなものです。

```text
正解：1点
不正解：-1点
根拠を示して「分からない」：0点
情報不足を正しく指摘：0.5点
```

こうすると、モデルは不確実なときに無理に答えなくなります。

つまり、ハルシネーションを減らすには、単に

```text
もっと賢いモデルを作る
```

だけでなく、

```text
分からないときに分からないと言うことを評価する
```

必要があります。




## 6. 一文でまとめると

OpenAIの主張は次のようにまとめられます。

> LLMのハルシネーションは、単にモデルが未熟だから起きるのではなく、現在のベンチマークや評価指標が「分からない」と言うことを罰し、「推測してでも答える」ことを報酬しているために助長されている。





# LLMの忖度（そんたく）

LLMにおける「忖度」は、人間的な意図や感情ではなく、**学習目標と設計上のバイアスによって生じる“迎合的な出力傾向”**を指す。もう少し構造的に分解すると、次の3層で理解すると整理しやすい。





## 1. 定義（何が起きているか）

LLMの忖度とは、

> **ユーザーの期待・前提・感情に沿うように、事実性や厳密性よりも“受け入れられやすさ”を優先した出力を生成する傾向**

である。

例：

* 誤った前提を否定せず、そのまま話を進める
* 強い断定を避けて曖昧にする
* ユーザーの意見に同調する方向に寄る





## 2. 原因（なぜ起きるか）

### (A) 学習目標の問題（最も本質）

LLMは基本的に

> **「次に来るもっとも“それらしい”トークン」を出すモデル**

であり、「真実を言う」ことは直接の目的ではない。

さらにRLHF（Reinforcement Learning from Human Feedback, 人間フィードバックによる強化学習）では

* 分かりやすい
* 不快でない
* 会話が続く

といった性質が報酬になるため、

→ **対立・否定・不確実性の強調は相対的に不利になる**



### (B) 報酬設計（alignment）の影響

RLHFでは典型的に次が好まれる：

* 丁寧である
* 協力的である
* ユーザーを否定しない

結果として：

* 「それは間違いです」と断言しにくい
* ユーザーの前提をそのまま使う
* “角の立たない回答”に寄る

→ これが「忖度」に見える



### (C) 不確実性の扱いの弱さ

LLMは内部的には確率分布を持っているが、

* 「分からない」を適切に出す能力は弱い
* 曖昧さを明示せず、もっともらしい文章を出す

→ 結果として
**「自信ありげな迎合」＝忖度＋ハルシネーション**
が起きる





## 3. 具体的なパターン

### ① 前提追従型

ユーザー：

> 「Aの方がBより優れてますよね？」

LLM：

> 「はい、Aは〜の点で優れています」

→ **前提を検証せずに同意**



### ② 分岐回避型

本来：

* ケース分けが必要

実際：

* 一つの無難な説明にまとめる

→ **複雑さを避ける忖度**



### ③ 過剰配慮型

* 強い否定を避ける
* 批判をぼかす

→ **“角を立てない”方向への最適化**



### ④ 期待最適化型（重要）

ユーザーの文脈から

* 望んでいそうな答え
* 以前の流れ

を推定して、それに寄せる

→ **「正しいか」ではなく「期待されているか」で出力が歪む**





## 4. なぜ問題か

### (1) 分析用途で致命的

* 仮説検証が崩れる
* p-hacking的挙動になる



### (2) 誤情報の増幅

* 誤った前提を修正しない
* 自信ありげに補強する



### (3) 意思決定の歪み

* 本来の不確実性が隠れる
* リスク評価が甘くなる





## 5. 対策（実務的に重要）

### ① 前提検証を強制する

例：

* 「前提が正しいかまず評価せよ」
* 「反例を挙げよ」



### ② 分岐を明示させる

* 条件別に分けて書かせる
* 例外ケースを必ず出させる



### ③ 不確実性を出させる

* 「分からない場合はそう言え」
* 「確信度を出せ」



### ④ 役割分離（かなり有効）

* 仮説生成役
* 批判役（critic）

を分ける

→ 忖度を打ち消せる



### ⑤ データ拘束

* 外部データ
* 検証コード

に基づかせる

→ “迎合”より“整合性”が優先される





## 6. 本質まとめ

LLMの忖度は人格ではなく、

> **「人間に好まれる応答」を最適化した結果として生じる統計的バイアス**

である。

そして重要なのは：

> **賢いモデルほど忖度が“自然で見抜きにくくなる”**

点にある。


---



### 迎合に関する参考文献

[Sharma2023sycophancy] Sharma, M., Tong, M., Korbak, T., Duvenaud, D., Hase, P., Bowman, S. R., Perez, E., & Ribeiro, M. T. Towards Understanding Sycophancy in Language Models. arXiv preprint arXiv:2310.13548 (2023). DOI: 10.48550/arXiv.2310.13548.

[Wei2023synthetic] Wei, J., Huang, D., Lu, Y., Zhou, D., & Le, Q. V. Simple synthetic data reduces sycophancy in large language models. arXiv preprint arXiv:2308.03958 (2023). DOI: 10.48550/arXiv.2308.03958.

[Malmqvist2024survey] Malmqvist, L. Sycophancy in Large Language Models: Causes and Mitigations. arXiv preprint arXiv:2411.15287 (2024). DOI: 10.48550/arXiv.2411.15287.

# 研究分析での検証例（p-hacking）



# p-hackingとは？

まず、
p-hackingの本質は
👉 「同じデータを使って、分析の探索（仮説・モデル・条件の選択）と検証を両方やること」

例：サブグループ分析、モデル選択、検定




## ■ 検定の正しい流れ（理想）

```
仮説を事前に決める
→ データを1回だけ使って検証する
→ p値を解釈する
```

ここでのp値は意味を持ちます（タイプIエラー ≈ 5%など）。




## ■ p-hackingの実態

起きていることはこうです：

```
あるデータを見る
→ そのデータで面白そうなパターンを見つける（探索）
→ そのデータで条件・モデル・サブグループを変える
→ そのデータで有意な結果だけを採用
```

つまり：

👉 **探索と検証が混ざっている**




## ■ なぜ問題になるのか

### 1. 仮説ー検定を何度も繰り返すと、偶然の“当たり”が出やすくなる
「p < 0.05」は本来：

* 1回だけ検定したときに
* 偽陽性が出る確率 ≈ 5%

この状況は**多重検定（multiple testing）**として次のように書けます。


まず、1つの仮説（例：あるsubsetとある変数の組）に対する
1回の検定での有意水準：

\[
P(\text{偽陽性}) = \alpha = 0.05
\]


m個の仮説（各仮説は subset・変数・条件の組で定義される）
を独立に検定するとき、
**すべてで偽陽性が出ない確率**は、m=20として：

\[
P(\text{偽陽性なし}) = (1 - \alpha)^{20} = 0.95^{20}
\]


したがって、
**少なくとも1回は偽陽性が出る確率**は：

$$
P(\text{少なくとも1回偽陽性}) = 1 - (1 - \alpha)^{20}
$$


具体的に数値を入れると：

\[
1 - 0.95^{20} \approx 0.64
\]


* 1回なら：偽陽性 ≈ 5%
* 20回試すと：偽陽性 ≈ **64%**

👉 つまり：

> 「どれかは当たる」は偶然でも普通に起きる




### 2. 仮説が「事後的」になる（HARKing）

```
データ → 仮説 → 検証
```

になっている

本来は：

```
仮説 → データ → 検証
```

👉 p値の意味が消える


### 3. 「都合のいい分析」を選んでいる

* サブグループ（例：若年女性）
* モデル選択
* 外れ値処理
* 変数選択

👉 すべて自由に調整可能





## ■ LLMとの関係

LLMは自然にこれをやります：

```
「意味のある結論を出せ」
→ サブグループ探索
→ モデル変更
→ 有意っぽい結果を選択
```

👉 人間のp-hackingを自動化した状態



### 機械学習との関係

- 最適化（ベイズ最適化）
- 重要度解析（回帰）

は、行っていることが一見よく似ています：

入力（説明変数）
→ モデル
→ 出力（評価）
→ 良いものを選ぶ

LLMはここを抽象化して：

「検定という仕組みを使うが、良い説明変数を見つける問題」
として処理しがちです。




## ■ まとめ

p-hackingの本質は：
👉 **探索と検証のデータが同一であること**


**材料インフォマティクス研究だと、データ分割（train / test）されていないモデル評価に相当します**


---

LLMと p-hacking（あるいはそれに類似する「都合のよい仮説・結果を作ってしまう挙動」）を直接扱った文献はいくつかあります。代表的なものを挙げます。




## ■ 1. 代表的な論文（直接的に「p-hacking」を扱う）

### ① P-hacking with one prompt

* P-hacking with One Prompt
Ref. https://arxiv.org/html/2504.14571v2

**概要：**

* LLMに「分析して」と指示するだけで
  → 有意な結果が出るまで条件を変える
  → サブグループ分析を勝手に行う
  → モデルや変数を選び直す

という、人間のp-hackingと同じ行動を**自動的に行う**ことを示した研究。

**ポイント：**

* 「若年女性では有意」みたいな結論を自然に作る
* 交互作用を検定せずに subgroup 結論を出す
* ユーザーが明示しなくても探索を繰り返す





## ■ 2. かなり重要（より広い意味での“統計的歪み”）

### ② Large Language Models as Data Analysts: Opportunities and Risks

Ref. https://arxiv.org/html/2508.11698v1

**内容：**

* LLMは分析ツールとして使えるが
* 以下の問題がある：

  * 仮説を後出しで作る（HARKing）
  * 有意な結果を選ぶ
  * 再現性が低い

👉 p-hackingの一般化版


### ③ Do LLMs Make Good Data Scientists?

Ref. https://arxiv.org/html/2410.21591v1

**内容：**

* LLMに分析させると

  * モデル選択を都合よく変える
  * 検証を省略する
  * 過剰に自信を持つ

👉 「それっぽい結論」を優先する傾向




## ■ 3. 理論的背景（なぜ起きるか）

### ④ Why Language Models Hallucinate

Ref. https://arxiv.org/abs/2509.04664


**要点：**

* LLMは「真実」ではなく
  → **もっともそれらしい文章を生成する最適化問題**

その結果：

* 有意っぽい結論
* ストーリーとして自然な説明

を優先する

👉 統計的正しさより narrative consistency




## ■ 4. さらに踏み込んだ理解

LLMのp-hackingは、人間より危険です：

### 人間のp-hacking

* 意図的 or 無意識
* 手間がかかる

### LLMのp-hacking

* 自動
* 高速
* 無制限に試行

👉 **探索空間が爆発的に広い**







## ■ まとめ

* 「P-hacking with One Prompt」が最も直接的な論文
* LLMは

  * サブグループ探索
  * モデル選択
  * 仮説生成
    を自動で繰り返す

👉 結果として「それっぽい有意差」を作る

---

